In [ ]:
# --- Imports ---
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV, learning_curve
)
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, accuracy_score,
    precision_recall_fscore_support
)
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

plt.rcParams.update({'figure.dpi': 120})

# --- Ruta fija para evitar "archivo no encontrado" ---
DATASET_PATH = "archive/Base.csv"   # <-- si lo tienes en otro lado, cámbialo aquí
print("Usando dataset en:", DATASET_PATH)


Usando dataset en: archive/Base.csv


In [2]:
# Carga el dataset en un dataframe de pandas y separa en características y etiquetas
def load_data1(path):
    """Carga el dataset desde un archivo CSV."""
    # Cargar el dataset
    df = pd.read_csv(path)
    X = df.drop(['fraud_bool'], axis=1)
    y = df['fraud_bool']
    return X, y

In [2]:
def load_data(path):
    df = pd.read_csv(path, low_memory=False)

    # Normaliza nombres para detección robusta (sin perder originales)
    original_cols = list(df.columns)
    norm_cols = [c.strip().lower() for c in original_cols]
    lower_to_original = {c.strip().lower(): c for c in original_cols}
    df.columns = norm_cols

    # Candidatos comunes para etiqueta
    candidates = ['fraud_bool','is_fraud','target','label','fraud','fraudulent','isfraud']
    target_col = next((c for c in candidates if c in df.columns), None)

    # Si no por nombre, intenta detectar una columna binaria 0/1
    if target_col is None:
        for c in df.columns:
            vals = pd.Series(df[c]).dropna().unique()
            if len(vals) <= 2 and set(pd.to_numeric(vals, errors='coerce').dropna().astype(int)).issubset({0,1}):
                target_col = c
                break
    if target_col is None:
        raise KeyError(f"No encuentro la columna objetivo. Columnas (primeras 20): {original_cols[:20]}")

    y = pd.to_numeric(df[target_col], errors='raise').astype(int)
    X = df.drop(columns=[target_col]).copy()

    # Quita posibles IDs/leaks
    leaks = ['uid','customer_id','eid','case_id','device_id','id','index']
    X = X.drop(columns=[c for c in leaks if c in X.columns], errors='ignore')

    # Restaura nombres originales legibles
    X.rename(columns={c: lower_to_original.get(c, c) for c in X.columns}, inplace=True)

    # Tipos y nulos
    for c in X.select_dtypes(include=['bool']).columns:
        X[c] = X[c].astype(int)
    for c in X.columns:
        if X[c].dtype.name in ['object','string']:
            X[c] = X[c].fillna('missing')
        else:
            X[c] = pd.to_numeric(X[c], errors='coerce')
            X[c] = X[c].fillna(X[c].median())

    num_cols = X.select_dtypes(include=['int64','float64','int32','float32']).columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]

    print(f"Target detectado: '{target_col}' | X: {X.shape} | y: {y.shape}")
    return X, y, num_cols, cat_cols

X, y, num_cols, cat_cols = load_data(DATASET_PATH)


Target detectado: 'fraud_bool' | X: (358837, 31) | y: (358837,)


In [ ]:
X, y = load_data1(DATASET_PATH)
num_cols = X.select_dtypes(include=['int64','float64','int32','float32']).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("Distribución clases train:\n", y_train.value_counts(normalize=True))

# Preprocesador
def make_preprocessor(num_cols, scale_numeric=True):
    trs = []
    if num_cols:
        trs.append(("num", StandardScaler() if scale_numeric else "passthrough", num_cols))
    return ColumnTransformer(trs)

preproc = make_preprocessor(num_cols, scale_numeric=True)
preproc


Distribución clases train:
 fraud_bool
0    0.988971
1    0.011029
Name: proportion, dtype: float64


NameError: name 'num_cols' is not defined

In [ ]:
# Funcion para graficar matriz de confusión
def plot_cm(cm, title='Matriz de confusión'):
    fig, ax = plt.subplots()
    im = ax.imshow(cm, interpolation='nearest')
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(cm.shape[1]), yticks=np.arange(cm.shape[0]),
        xticklabels=['No Fraud','Fraud'], yticklabels=['No Fraud','Fraud'],
        ylabel='True', xlabel='Pred', title=title
    )
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], 'd'), ha="center", va="center")
    plt.tight_layout(); plt.show()


In [ ]:
svm = Pipeline([
    ('pre', preproc),
    ('clf', SVC(kernel='rbf', probability=True, class_weight='balanced'))
])

# Para pruebas rápidas puedes reducir el grid; para resultado final deja/expande
param_grid = {
    'clf__C': [0.5, 1, 2, 5],
    'clf__gamma': ['scale', 0.1, 0.01, 0.001]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(svm, param_grid, cv=cv, scoring='f1', n_jobs=-1, refit=True, verbose=0)
grid.fit(X_train, y_train)

print("[SVM] Mejores params:", grid.best_params_)
y_pred_svm  = grid.predict(X_test)
y_proba_svm = grid.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred_svm, digits=4))
print("AUC:", roc_auc_score(y_test, y_proba_svm))
plot_cm(confusion_matrix(y_test, y_pred_svm), 'Matriz — SVM RBF')


In [ ]:
mlp = Pipeline([
    ('pre', preproc),
    ('clf', MLPClassifier(max_iter=100, random_state=42))
])

param_dist = {
    'clf__hidden_layer_sizes': [(64,), (128,), (64,32), (128,64)],
    'clf__alpha': [1e-5, 1e-4, 1e-3, 1e-2]
}

rnd = RandomizedSearchCV(
    mlp, param_dist, n_iter=6, cv=cv, scoring='f1',
    n_jobs=-1, random_state=42, refit=True, verbose=0
)
rnd.fit(X_train, y_train)

print("[MLP] Mejores params:", rnd.best_params_)
y_pred_mlp  = rnd.predict(X_test)
y_proba_mlp = rnd.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred_mlp, digits=4))
print("AUC:", roc_auc_score(y_test, y_proba_mlp))
plot_cm(confusion_matrix(y_test, y_pred_mlp), 'Matriz — MLP')


In [ ]:
def plot_learning_curve(pipe, X, y, title):
    sizes, train_scores, val_scores = learning_curve(
        pipe, X, y, cv=cv, scoring='f1', n_jobs=-1, train_sizes=np.linspace(0.2,1.0,5)
    )
    fig, ax = plt.subplots()
    ax.plot(sizes, train_scores.mean(axis=1), marker='o', label='Train F1')
    ax.plot(sizes, val_scores.mean(axis=1),  marker='s', label='CV F1')
    ax.set_xlabel('Tamaño de entrenamiento'); ax.set_ylabel('F1'); ax.set_title(title); ax.legend()
    plt.tight_layout(); plt.show()

plot_learning_curve(grid.best_estimator_, X_train, y_train, 'Learning Curve — SVM RBF')
plot_learning_curve(rnd.best_estimator_,  X_train, y_train, 'Learning Curve — MLP')


In [ ]:
def summarize(name, y_true, y_pred, y_proba):
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    p1, r1, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary', pos_label=1, zero_division=0
    )
    return {
        'modelo': name,
        'accuracy': accuracy_score(y_true, y_pred),
        'f1_weighted': rep['weighted avg']['f1-score'],
        'auc': roc_auc_score(y_true, y_proba),
        'precision_1': p1,
        'recall_1': r1,
        'f1_1': f1
    }

summary = []
summary.append(summarize('Baseline LR', y_test, y_pred_b,  y_proba_b))
summary.append(summarize('SVM RBF',     y_test, y_pred_svm, y_proba_svm))
summary.append(summarize('MLP',         y_test, y_pred_mlp, y_proba_mlp))

pd.DataFrame(summary)
